<a href="https://colab.research.google.com/github/nisal-eng/Statistical-Learning-e22206/blob/main/Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Advanced Bayesian Parameter Estimation, Tracking, & Probabilistic Clustering**

1. Bayesian Estimation of a User Ability Parameter from Item ResponsesVisualizing the MechanicsThe two-parameter logistic (2PL) item response model defines the probability of a correct response as:$$P(X_j = 1 \mid \theta) = \frac{1}{1 + e^{-\alpha_j(\theta - \beta_j)}}$$The parameter $\beta_j$ acts as a horizontal shift parameter. When $\theta = \beta_j$, the exponent becomes 0, meaning $P(X_j = 1 \mid \theta) = 0.5$. Increasing $\beta_j$ shifts the entire response curve to the right along the ability axis. This means a user requires a higher level of latent ability $\theta$ to maintain the same probability of scoring a correct answer.

In [1]:
import numpy as np
import plotly.graph_objects as go

theta = np.linspace(-4, 4, 200)

# 2PL function
def p_2pl(theta, alpha, beta):
    return 1 / (1 + np.exp(-alpha * (theta - beta)))

fig = go.Figure()
# Pair 1: Alpha = 1.5, vary Beta
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 1.5, -1), name="α=1.5, β=-1 (Easy)"))
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 1.5, 0), name="α=1.5, β=0 (Medium)"))
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 1.5, 1), name="α=1.5, β=1 (Hard)"))
# Pair 2: Distinct Alpha value
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 0.6, 0), name="α=0.6, β=0 (Low Discrim)", line=dict(dash='dash')))

fig.update_layout(
    title="2PL Item Response Theory Characteristic Curves",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response",
    template="plotly_white"
)
fig.show()

Sequential Likelihood ContributionFor a single isolated response $x_j \in \{0, 1\}$ at step $j$:$$L(x_j \mid \theta) = [P(X_j = 1 \mid \theta)]^{x_j} [1 - P(X_j = 1 \mid \theta)]^{1 - x_j}$$Given the conditional independence assumption of Item Response Theory, the joint likelihood function for the entire running vector $X^{(1:j)}$ is the product of the individual Bernoulli trials:$$L(X^{(1:j)} \mid \theta) = \prod_{i=1}^j [P(X_i = 1 \mid \theta)]^{x_i} [1 - P(X_i = 1 \mid \theta)]^{1 - x_i}$$Mathematical Formulation of the Running UpdateUsing Bayes' theorem sequentially, the posterior distribution at step $j-1$ becomes the prior distribution for step $j$. The recursive relationship is stated as:$$p(\theta \mid X^{(1:j)}) \propto p(\theta \mid X^{(1:j-1)}) \cdot L(x_j \mid \theta)$$Dynamic Shifting MechanicsWhen a user correctly answers ($x_j = 1$) a highly difficult item (large $\beta_j$), the likelihood function $P(X_j = 1 \mid \theta)$ is an S-curve that remains very low for small values of $\theta$ and rises sharply only when $\theta$ approaches or exceeds $\beta_j$. Multiplying the previous posterior density $p(\theta \mid X^{(1:j-1)})$ by this heavily right-skewed likelihood suppresses the density values across the lower ability spectrum. This moves the peak (mode) of the updated posterior distribution significantly to the right, rewarding the system's belief in the user's ability.Tracking Certainty and SharpnessThe item discrimination parameter $\alpha_j$ dictates the steepness of the item characteristic curve, which directly affects the variance of the posterior distribution:Large $\alpha_j$: The likelihood function changes rapidly from 0 to 1 over a narrow range of $\theta$. This injects a high-precision informational signal into the recursive multiplication, narrowing the posterior distribution and significantly increasing its sharpness (reducing variance).Small $\alpha_j$: The likelihood curve is flat and uninformative. Multiplying by this flat curve leaves the shape of the existing prior almost unchanged, providing minimal reduction in variance.Numerical Implementation of a Running GridTo evaluate this non-conjugate update without analytical integration, we establish a fixed, equally spaced grid vector $\Theta = [\theta_1, \theta_2, \dots, \theta_M]$ over a closed interval such as $[-4, 4]$.Initialize an array representing the prior evaluation: $P_0 = [p(\theta_1), p(\theta_2), \dots, p(\theta_M)]$, where $p(\theta_m) = \frac{1}{\sqrt{2\pi}}e^{-\theta_m^2/2}$.Upon observing a response $x_j$ for an item parameterized by $(\alpha_j, \beta_j)$, compute the likelihood vector $L_j$ across the grid: $L_{j,m} = [P(X_j = 1 \mid \theta_m)]^{x_m} [1 - P(X_j = 1 \mid \theta_m)]^{1 - x_m}$.Perform element-wise multiplication to update the unnormalized density: $\tilde{P}_{j,m} = P_{j-1,m} \times L_{j,m}$.Normalize the grid computationally via the trapezoidal rule to guarantee the total area sums to 1:$$\text{Area} = \frac{\Delta \theta}{2} \sum_{m=1}^{M-1} (\tilde{P}_{j,m} + \tilde{P}_{j,m+1})$$$$P_{j,m} = \frac{\tilde{P}_{j,m}}{\text{Area}}$$Evaluating Convergence over the Timeline

In [2]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
N = 50
theta_true = 1.2
grid = np.linspace(-4, 4, 1000)
d_theta = grid[1] - grid[0]

# Standard Normal Prior
posterior_grid = np.exp(-grid**2 / 2) / np.sqrt(2 * np.pi)

# Generate Random Item Parameters
alphas = np.random.uniform(0.5, 2.5, N)
betas = np.random.uniform(-2.0, 2.0, N)

est_mean = []
est_map = []

for j in range(N):
    # True probability of correct answer
    p_true = 1 / (1 + np.exp(-alphas[j] * (theta_true - betas[j])))
    xj = 1 if np.random.uniform(0, 1) < p_true else 0

    # Likelihood over grid
    p_grid = 1 / (1 + np.exp(-alphas[j] * (grid - betas[j])))
    likelihood = p_grid if xj == 1 else (1 - p_grid)

    # Update and Normalize
    posterior_grid = posterior_grid * likelihood
    area = np.trapz(posterior_grid, grid)
    posterior_grid /= area

    # Compute Estimates
    current_mean = np.trapz(posterior_grid * grid, grid)
    current_map = grid[np.argmax(posterior_grid)]

    est_mean.append(current_mean)
    est_map.append(current_map)

# Visualization
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(1, N+1)), y=est_mean, name="Posterior Mean (E[θ])", mode='lines+markers'))
fig.add_trace(go.Scatter(x=list(range(1, N+1)), y=est_map, name="MAP Estimate", mode='lines+markers'))
fig.add_trace(go.Scatter(x=[1, N], y=[theta_true, theta_true], name="True Ability (θ_true)", line=dict(dash='dash', color='black')))
fig.update_layout(title="Sequential IRT Parameter Tracking Convergence", xaxis_title="Item Number (j)", yaxis_title="Estimated Ability", template="plotly_white")
fig.show()

/tmp/ipykernel_973/879358793.py:31: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_973/879358793.py:35: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



AnalysisAs the item step count $j$ increases, the distance between both sequential estimators and the true ability $\theta_{\text{true}} = 1.2$ shrinks. Initial tracking steps oscillate significantly due to the strong influence of individual random coin flips. However, as evidence accumulates, the posterior distribution narrows around the true parameter value. This narrowing implies that the platform's variance is shrinking, signaling a statistical increase in measurement confidence.2. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates1. Structural Probability and Properties

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

p_axis = np.linspace(0, 1, 500)

fig = go.Figure()
fig.add_trace(go.Scatter(x=p_axis, y=stats.beta.pdf(p_axis, 1, 1), name="α=1, β=1 (Uniform)"))
fig.add_trace(go.Scatter(x=p_axis, y=stats.beta.pdf(p_axis, 2, 8), name="α=2, β=8 (Right-skewed)"))
fig.add_trace(go.Scatter(x=p_axis, y=stats.beta.pdf(p_axis, 8, 2), name="α=8, β=2 (Left-skewed)"))
fig.update_layout(title="Beta Distribution Prior Configurations", xaxis_title="CTR Parameter (p)", yaxis_title="Density", template="plotly_white")
fig.show()

The parameters $\alpha$ and $\beta$ act as pseudo-counts for successes (clicks) and failures (non-clicks). When $\alpha = \beta$, the distribution is symmetric around $0.5$. When $\beta$ is greater than $\alpha$, the center of mass shifts left (representing a low expected conversion rate). Conversely, when $\alpha$ is greater than $\beta$, the distribution is pushed toward 1, reflecting an optimistic prior belief.2. Sequential Likelihood and Joint HistoryThe likelihood contribution of an isolated outcome $y_j \in \{0, 1\}$ given click probability $p$ is modeled as a Bernoulli trial:$$L(y_j \mid p) = p^{y_j}(1 - p)^{1 - y_j}$$Assuming independence conditional on $p$, the joint likelihood function for the running history vector $Y^{(1:j)}$ is:$$L(Y^{(1:j)} \mid p) = \prod_{i=1}^j p^{y_i}(1 - p)^{1 - y_i} = p^{\sum y_i} (1 - p)^{j - \sum y_i}$$3. Closed-Form Analytical Updates (Conjugacy)By Bayes' theorem, the posterior density at step $j$ satisfies:$$p(p \mid Y^{(1:j)}) \propto p(p \mid Y^{(1:j-1)}) \cdot L(y_j \mid p)$$Let the prior distribution at step $j$ be distributed as $\text{Beta}(\alpha_{j-1}, \beta_{j-1})$:$$p(p \mid Y^{(1:j)}) \propto \left[ \frac{1}{\text{B}(\alpha_{j-1}, \beta_{j-1})} p^{\alpha_{j-1} - 1}(1 - p)^{\beta_{j-1} - 1} \right] \cdot \left[ p^{y_j}(1 - p)^{1 - y_j} \right]$$$$p(p \mid Y^{(1:j)}) \propto p^{(\alpha_{j-1} + y_j) - 1} (1 - p)^{(\beta_{j-1} + 1 - y_j) - 1}$$This functional form matches the kernel of a Beta distribution. This proves that the Beta family is conjugate to the Bernoulli likelihood. The exact closed-form algebraic update parameters are:$$\alpha_j = \alpha_{j-1} + y_j$$$$\beta_j = \beta_{j-1} + (1 - y_j)$$The exact Posterior Mean $E[p \mid Y^{(1:j)}]$ under this formulation is:$$E[p \mid Y^{(1:j)}] = \frac{\alpha_j}{\alpha_j + \beta_j}$$4. Dynamic Shifting MechanicsObserved Click ($y_j = 1$): Directly increments $\alpha$, shifting the peak of the probability density function to the right.Observed Non-click ($y_j = 0$): Increments $\beta$, shifting the density peak to the left.Unlike the 2PL IRT model, which requires numerical integration across an artificial grid at every single iteration, this conjugate framework updates its parameters using simple addition. The exact updated probability distribution is calculated instantaneously without numerical approximations.5. Running Point EstimatorsAt any step $j$, the exact point estimators derived from the updated shape parameters are:Running Posterior Mean: $\hat{p}_{\text{Mean}} = \frac{\alpha_j}{\alpha_j + \beta_j}$Running Maximum A Posteriori (MAP): $\hat{p}_{\text{MAP}} = \frac{\alpha_j - 1}{\alpha_j + \beta_j - 2} \quad (\text{for } \alpha_j, \beta_j > 1)$6. Performance Tracking and Convergence Analysis

In [4]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
T = 500
p_true = 0.18

# Uninformative Base Prior
alpha_curr, beta_curr = 1.0, 1.0

means = []
maps = []

for t in range(1, T + 1):
    yt = 1 if np.random.uniform(0, 1) < p_true else 0

    # Analytical Update
    alpha_curr += yt
    beta_curr += (1 - yt)

    # Store Estimates
    means.append(alpha_curr / (alpha_curr + beta_curr))
    maps.append((alpha_curr - 1) / (alpha_curr + beta_curr - 2) if (alpha_curr + beta_curr) > 2 else 0.5)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(1, T+1)), y=means, name="Posterior Mean"))
fig.add_trace(go.Scatter(x=list(range(1, T+1)), y=maps, name="MAP Estimate"))
fig.add_trace(go.Scatter(x=[1, T], y=[p_true, p_true], name="True CTR", line=dict(dash='dash', color='black')))
fig.update_layout(title="Beta-Binomial Sequential CTR Tracking", xaxis_title="Impressions (t)", yaxis_title="Estimated CTR", template="plotly_white")
fig.show()

AnalysisAs the number of impressions $t$ approaches $500$, the sample size dominates the prior distribution. The distance between the estimators and the true parameter value converges toward zero. This rapid tracking shows that as data accumulates, initial prior assumptions are quickly overridden by the empirical evidence, shrinking the variance of the posterior distribution.3. Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates1. Prior Belief Boundaries

In [5]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

phi_grid = np.linspace(0.1, 1.0, 500)
# Beta parameters
alpha_p, beta_p = 12.0, 2.0
prior_density = stats.beta.pdf(phi_grid, alpha_p, beta_p)

fig = go.Figure()
fig.add_trace(go.Scatter(x=phi_grid, y=prior_density, name="Beta(12, 2) Prior"))
fig.update_layout(title="Initial Structural Stiffness Prior Belief", xaxis_title="Stiffness Factor (φ)", yaxis_title="Density", template="plotly_white")
fig.show()

The analytical expectation of the prior distribution is:$$E[\phi] = \frac{\alpha}{\alpha + \beta} = \frac{12}{12 + 2} = \frac{12}{14} \approx 0.857$$This distribution serves as an effective prior for structural safety monitoring because its probability mass is concentrated near 1.0. This reflects the engineering assumption that the component is highly likely to be healthy and undamaged when deployed, while still allowing for lower values if supported by the sensor data.2. Structural Likelihood FormulationThe degradation physics states that $S_k = S_0 \phi e^{\epsilon_k}$, where $\epsilon_k \sim \mathcal{N}(0, \sigma^2)$. Rearranging for the error term gives $\log(S_k / (S_0 \phi)) = \epsilon_k$. Applying a change of variables, the probability density function for the continuous sensor measurement $S_k$ is:$$L(S_k \mid \phi) = \frac{1}{S_k \sigma \sqrt{2\pi}} \exp \left( -\frac{(\log S_k - \log(S_0 \phi))^2}{2\sigma^2} \right)$$Assuming independent measurement errors over time, the joint likelihood for the running history vector $S^{(1:k)}$ is:$$L(S^{(1:k)} \mid \phi) = \prod_{i=1}^k \frac{1}{S_i \sigma \sqrt{2\pi}} \exp \left( -\frac{(\log S_i - \log(S_0 \phi))^2}{2\sigma^2} \right)$$3. Mathematical Formulation of the Non-Conjugate Grid UpdateAn exact closed-form analytical solution does not exist because multiplying a Beta distribution (a polynomial form $\phi^{\alpha-1}(1-\phi)^{\beta-1}$) by a Log-Normal distribution (where $\phi$ is inside a logarithm within an exponent) results in a non-standard kernel that cannot be simplified into known parametric distributions.The recursive update formula is defined as:$$p(\phi \mid S^{(1:k)}) \propto p(\phi \mid S^{(1:k-1)}) \cdot \frac{1}{S_k \sigma \sqrt{2\pi}} \exp \left( -\frac{(\log S_k - \log(S_0 \phi))^2}{2\sigma^2} \right)$$4. Running Point EstimatesBecause an analytical solution is unavailable, we calculate point estimates using numerical integration over the bounded domain $[0.1, 1.0]$:Running Posterior Mean:$$\hat{\phi}_{\text{Mean}} = \int_{0.1}^{1.0} \phi \cdot p(\phi \mid S^{(1:k)}) \, d\phi$$Running Maximum A Posteriori (MAP):$$\hat{\phi}_{\text{MAP}} = \arg\max_{\phi \in [0.1, 1.0]} p(\phi \mid S^{(1:k)})$$5. Algorithmic Grid Approximation and NormalizationDiscretize the physical bounded range $[0.1, 1.0]$ into $M$ equally spaced nodes: $\Phi = [\phi_1, \phi_2, \dots, \phi_M]$.Initialize the prior array using the Beta distribution density evaluated at each node: $P_{0, m} = \text{Beta.pdf}(\phi_m \mid 12, 2)$.When a new sensor reading $S_k$ arrives, evaluate the log-normal likelihood across all grid positions:$$L_{k, m} = \frac{1}{S_k \sigma \sqrt{2\pi}} \exp \left( -\frac{(\log S_k - \log(S_0 \phi_m))^2}{2\sigma^2} \right)$$Compute the unnormalized updated density by element-wise multiplication: $\tilde{P}_{k, m} = P_{k-1, m} \times L_{k, m}$.Apply the trapezoidal rule across the grid to calculate the normalizing constant:$$\text{Area} = \frac{\phi_M - \phi_1}{2(M-1)} \sum_{m=1}^{M-1} (\tilde{P}_{k, m} + \tilde{P}_{k, m+1})$$Normalize the grid values: $P_{k, m} = \frac{\tilde{P}_{k, m}}{\text{Area}}$.6. Performance Tracking and Degradation Convergence Analysis

In [7]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(42)
K = 60
phi_true = 0.68
S0 = 200.0
sigma = 0.15

grid_phi = np.linspace(0.1, 1.0, 1000)

# Initialize prior array
post_curves = stats.beta.pdf(grid_phi, 12, 2)
area_init = np.trapezoid(post_curves, grid_phi)
post_curves /= area_init

milestones = [0, 5, 15, 30, 60]
saved_curves = {0: post_curves.copy()}

history_mean = []
history_map = []

for k in range(1, K + 1):
    # Simulate data based on true log-normal physics
    sk = S0 * phi_true * np.exp(np.random.normal(0, sigma))

    # Calculate likelihood across grid
    likelihood = (1.0 / (sk * sigma * np.sqrt(2 * np.pi))) * np.exp(
        -((np.log(sk) - np.log(S0 * grid_phi)) ** 2) / (2 * sigma ** 2)
    )

    # Numerical Update
    post_curves = post_curves * likelihood
    norm_factor = np.trapezoid(post_curves, grid_phi)
    post_curves /= norm_factor

    if k in milestones:
        saved_curves[k] = post_curves.copy()

    # Store point estimates
    history_mean.append(np.trapezoid(post_curves * grid_phi, grid_phi))
    history_map.append(grid_phi[np.argmax(post_curves)])

# Plot 1: Evolution of Density Profiles
fig1 = go.Figure()
for m in milestones:
    fig1.add_trace(go.Scatter(x=grid_phi, y=saved_curves[m], name=f"Step {m}"))
fig1.update_layout(title="Posterior Density Evolution over Inspection Timeline", xaxis_title="Stiffness Efficiency Factor (φ)", yaxis_title="Density", template="plotly_white")
fig1.show()

# Plot 2: Convergence Estimation Profiles
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=list(range(1, K+1)), y=history_mean, name="Inferred Mean (E[φ])"))
fig2.add_trace(go.Scatter(x=list(range(1, K+1)), y=history_map, name="Inferred MAP"))
fig2.add_trace(go.Scatter(x=[1, K], y=[phi_true, phi_true], name="True Factor (0.68)", line=dict(dash='dash', color='black')))
fig2.update_layout(title="Structural Health Parameter Estimation Tracking Trajectory", xaxis_title="Inspection Step (k)", yaxis_title="Efficiency Value", template="plotly_white")
fig2.show()

/tmp/ipykernel_973/3902194323.py:15: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_973/3902194323.py:35: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_973/3902194323.py:42: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



AnalysisThe system successfully overrides the optimistic prior belief within approximately 10 to 15 sensor iterations, shifting its probability mass to focus on the true underlying structural damage state ($\phi = 0.68$). As more measurements are processed, the width of the posterior distribution narrows significantly. This narrowing reflects decreasing uncertainty, allowing operators to reliably determine when structural safety thresholds have been crossed.4. Gaussian Mixture Clustering as Conditional UpdatingDeriving the Marginal DensityBy the law of total probability, we obtain the marginal distribution of an observed data point $x_i$ by summing the joint distribution over all possible discrete states of the latent cluster assignment variable $z_i$:$$p(x_i \mid \Theta) = \sum_{k=1}^K p(x_i, z_i = k \mid \Theta)$$$$p(x_i \mid \Theta) = \sum_{k=1}^K p(z_i = k \mid \Theta) \cdot p(x_i \mid z_i = k, \Theta)$$$$p(x_i \mid \Theta) = \sum_{k=1}^K \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$$This expression is called a Gaussian mixture density because it is a convex combination (a weighted sum where the weights are non-negative and sum to 1) of $K$ distinct multivariate Gaussian distributions.Deriving the Posterior Cluster ProbabilityApplying Bayes' rule for a fixed observation $x_i$, the conditional probability that it belongs to cluster $k$ is:$$p(z_i = k \mid x_i, \Theta) = \frac{p(z_i = k \mid \Theta) \cdot p(x_i \mid z_i = k, \Theta)}{p(x_i \mid \Theta)}$$Substituting the prior probabilities $\phi_k$ and the Gaussian component densities yields:$$\gamma_{ik} = \frac{\phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$$The quantity $\gamma_{ik}$ represents the responsibility that cluster $k$ takes for generating data point $x_i$. It is interpreted as a posterior probability because it updates our initial prior belief $\phi_k$ after incorporating the empirical evidence provided by the location of $x_i$.One-Hot Encoding of the Latent Cluster VariableLet $z_i$ be represented as a binary one-hot vector $z_i = [z_{i1}, z_{i2}, \dots, z_{iK}]^T$. Since $z_i$ must belong to exactly one cluster, only one element can be equal to 1, while all others are 0. Therefore:$$E[z_{ik} \mid x_i, \Theta] = 1 \cdot p(z_{ik} = 1 \mid x_i, \Theta) + 0 \cdot p(z_{ik} = 0 \mid x_i, \Theta) = p(z_i = k \mid x_i, \Theta) = \gamma_{ik}$$The conditional expectation vector of the one-hot encoded latent variable matches the soft assignment vector:$$E[z_i \mid x_i, \Theta] = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$From Soft Assignment to Hard ClusteringSoft Clustering: Assigns a continuous probability distribution across all $K$ dimensions ($\gamma_{i1}, \dots, \gamma_{iK}$), capturing the inherent ambiguity for points located near cluster boundaries.Hard Clustering: Makes a definitive assignment to a single cluster by selecting the index with the highest posterior probability: $c_i = \arg\max_k \gamma_{ik}$. This discards information about assignment uncertainty.Conditional Expectation of the Observation Given the ClusterThe conditional expectation of the data point given that it belongs to cluster $k$ is:$$E[x_i \mid z_i = k] = \int x_i \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \, dx_i = \mu_k$$This defines $\mu_k$ as the center of mass for that component.$E[z_i \mid x_i, \Theta]$ maps an observed data point to its posterior cluster membership probabilities.$E[x_i \mid z_i = k, \Theta]$ defines the expected location of an observation given its cluster assignment.The Complete-Data LikelihoodIf the latent cluster assignments $z_{ik}$ were observed, the joint density for the complete data would be:$$p(X, Z \mid \Theta) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$Taking the natural logarithm simplifies the product into sums:$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$If the indicators $z_{ik}$ were known, we could maximize this log-likelihood directly for each cluster independently. We would simply group the data points by their assigned cluster and calculate the sample mean and variance for each group using standard maximum likelihood estimation.The EM InterpretationIn practice, the latent variables $z_{ik}$ are unobserved. The Expectation-Maximization (EM) algorithm handles this by taking the conditional expectation of the complete-data log-likelihood with respect to the posterior distribution of the latent variables, given the current parameter estimates $\Theta^{\text{old}}$:$$Q(\Theta, \Theta^{\text{old}}) = E_{Z \mid X, \Theta^{\text{old}}}[\ell_c]$$$$Q(\Theta, \Theta^{\text{old}}) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$The E-step can be viewed as a conditional update step because it uses the current parameters to compute the posterior probabilities $\gamma_{ik}$, updating our estimate of cluster memberships across the entire dataset.Parameter UpdatesDuring the M-step, we maximize the expected complete-data log-likelihood $Q$ with respect to the parameters. To update the mixture weights $\phi_k$ subject to the constraint $\sum_{k=1}^K \phi_k = 1$, we construct the Lagrangian function:$$\mathcal{L}(\phi, \lambda) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \log \phi_k + \lambda \left( 1 - \sum_{k=1}^K \phi_k \right)$$Taking the partial derivative with respect to $\phi_k$ and setting it to 0 gives:$$\frac{\partial \mathcal{L}}{\partial \phi_k} = \sum_{i=1}^n \frac{\gamma_{ik}}{\phi_k} - \lambda = 0 \implies \phi_k = \frac{\sum_{i=1}^n \gamma_{ik}}{\lambda}$$Summing both sides over $k$ shows that $\lambda = n$, which yields the update rule:$$\phi_k^{\text{new}} = \frac{\sum_{i=1}^n \gamma_{ik}}{n}$$For the cluster means $\mu_k$, we maximize the term containing the Gaussian density:$$\nabla_{\mu_k} Q = \sum_{i=1}^n \gamma_{ik} \Sigma_k^{-1}(x_i - \mu_k) = 0 \implies \mu_k^{\text{new}} = \frac{\sum_{i=1}^n \gamma_{ik} x_i}{\sum_{i=1}^n \gamma_{ik}}$$Similarly, maximizing with respect to the covariance matrix $\Sigma_k$ results in:$$\Sigma_k^{\text{new}} = \frac{\sum_{i=1}^n \gamma_{ik}(x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T}{\sum_{i=1}^n \gamma_{ik}}$$The responsibility $\gamma_{ik}$ acts as a fractional weight. Data points with a high posterior probability of belonging to cluster $k$ contribute significantly to the updated estimates for that cluster's mean and covariance, while points with low probability have minimal impact.InterpretationGaussian Mixture Model (GMM) clustering operates as an iterative process of conditional updating:Prior Probabilities: The mixture weights $\phi_k$ represent the prior probability of an observation belonging to cluster $k$ before examining its features.Compatibility: The Gaussian density $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ measures how compatible an observed point $x_i$ is with the current shape and location of cluster $k$.Posterior Responsibilities: The responsibility $\gamma_{ik}$ combines the prior and the likelihood via Bayes' rule, yielding the updated posterior probability of cluster membership.Parameter Optimization: The M-step updates the cluster shapes ($\mu_k, \Sigma_k$) using these posterior probabilities as weights, shifting the cluster definitions to better fit the data.This iterative loop demonstrates that GMM clustering is a probabilistic framework built on the conditional expectations of latent variables.5. Computational Simulation and Out-of-Sample ValidationThe complete python implementation using the CC GENERAL.csv dataset from Kaggle is structured below:

In [9]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
import plotly.express as px

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.model = GaussianMixture(
            n_components=self.n_components,
            random_state=self.random_state,
            init_params='kmeans'
        )

    def prepare_data(self, df):
        # Extract features and handle missing records
        data = df[['PURCHASES', 'CREDIT_LIMIT']].dropna()

        # Split into training and validation sets
        train_data, test_data = train_test_split(
            data, test_size=0.2, random_state=self.random_state
        )

        # Standardize features
        self.X_train = self.scaler.fit_transform(train_data)
        self.X_test = self.scaler.transform(test_data)
        return self.X_train, self.X_test

    def fit_em(self):
        self.model.fit(self.X_train)
        print(f"Convergence Status: {self.model.converged_}")
        print(f"Iterations Required: {self.model.n_iter_}")

    def evaluate_test(self):
        score = self.model.score(self.X_test)
        print(f"Average Out-of-Sample Log-Likelihood Score: {score:.4f}")
        return score

    def plot_density_heatmap(self):
        fig = px.density_heatmap(
            x=self.X_train[:, 0], y=self.X_train[:, 1],
            marginal_x="histogram", marginal_y="histogram",
            labels={'x': 'Scaled Purchases', 'y': 'Scaled Credit Limit'},
            title="Empirical 2D Density Heatmap (Training Set Data)"
        )
        fig.update_layout(template="plotly_white")
        return fig

    def _generate_contour_grid(self):
        # Helper to construct grid mesh coordinates for contour maps
        x_min, x_max = self.X_train[:, 0].min() - 0.5, self.X_train[:, 0].max() + 0.5
        y_min, y_max = self.X_train[:, 1].min() - 0.5, self.X_train[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        # Compute responsibilities across the grid
        responsibilities = self.model.predict_proba(grid_points)
        max_resp = responsibilities.max(axis=1).reshape(xx.shape)
        return xx, yy, max_resp

    def plot_assignments(self, data_type="train"):
        xx, yy, max_resp = self._generate_contour_grid()
        X_pts = self.X_train if data_type == "train" else self.X_test
        labels = self.model.predict(X_pts)

        fig = go.Figure()
        # Overlay continuous posterior uncertainty contour field
        fig.add_trace(go.Contour(
            x=np.linspace(xx.min(), xx.max(), 200),
            y=np.linspace(yy.min(), yy.max(), 200),
            z=max_resp, colorscale='Viridis', opacity=0.35,
            colorbar=dict(title="Max Resp (γ_ik)")
        ))
        # Overlay hard discrete scatter points
        fig.add_trace(go.Scatter(
            x=X_pts[:, 0], y=X_pts[:, 1],
            mode='markers',
            marker=dict(color=labels, colorscale='Plotly3', size=5),
            name="Data Observations"
        ))
        title_text = f"GMM Bounds vs Max Responsibility Overlay ({data_type.capitalize()} Set)"
        fig.update_layout(title=title_text, xaxis_title="Scaled Purchases", yaxis_title="Scaled Credit Limit", template="plotly_white")
        return fig

# Dummy execution block to demonstrate validation layout patterns
if __name__ == "__main__":
    # Create synthetic dataset to simulate the CC GENERAL data schema
    np.random.seed(42)
    syn_data = np.random.multivariate_normal([500, 2000], [[100000, 20000], [20000, 500000]], 1000)
    syn_data2 = np.random.multivariate_normal([4000, 8000], [[500000, -10000], [-10000, 1000000]], 500)
    full_syn = np.vstack([syn_data, syn_data2])
    mock_df = pd.DataFrame(full_syn, columns=['PURCHASES', 'CREDIT_LIMIT'])

    segmenter = GMMFinancialSegmenter(n_components=3)
    segmenter.prepare_data(mock_df)
    segmenter.fit_em()
    segmenter.evaluate_test()

    # Generate the interactive visualization objects
    fig_heatmap = segmenter.plot_density_heatmap()
    fig_train = segmenter.plot_assignments("train")
    fig_test = segmenter.plot_assignments("test")

    # Display figures
    fig_heatmap.show()
    fig_train.show()
    fig_test.show()

Convergence Status: True
Iterations Required: 4
Average Out-of-Sample Log-Likelihood Score: -0.8443


Evaluation & Analytical TakeawaysThe background contour maps display a continuous gradient that represents the soft assignment expectation vector $E[z_i \mid x_i, \Theta]$. In regions close to the cluster centers, the background color shows a high maximum responsibility ($\gamma_{ik} \approx 1.0$), creating flat plateaus of high certainty.As you move toward the boundaries between clusters, the maximum responsibility drops significantly, approaching values like $0.5$ or $0.33$. This color gradient visualizes the soft clustering framework, illustrating the transition zones where data points carry substantial uncertainty regarding their cluster membership.